In [1]:
# import 
import pandas as pd
import numpy as np
import sqlite3


In [2]:
from sqlalchemy import create_engine

In [ ]:
warehouses = pd.read_csv('C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/warehouses.csv' )
warehouses.head()

In [4]:
USER = "root"
PAPSSWORD = "Abhi8383055393"
HOST = "localhost"
port = "3306"
datb = "ecommmerce_analysis"



In [5]:
# mysql
connect_str = f"mysql+pymysql://{USER}:{PAPSSWORD}@{HOST}:{port}/{datb}"

In [6]:
mysql_conn = create_engine(connect_str)

In [ ]:
# checking the database
warehouses.info()

In [ ]:
warehouses[warehouses['city'].isna()== True]

In [9]:
warehouses[['state', 'city']]
list(warehouses[['state', 'city']])

['state', 'city']

In [10]:
city_map = (
    warehouses
    .dropna(subset=['state','city'])
    .drop_duplicates(subset='city')
    .set_index('state')['city']
)

In [11]:
state_map = (warehouses.drop_duplicates('state').set_index('state')['city'])

In [12]:
warehouses['city'] = warehouses['city'].fillna(warehouses['state'].map(state_map))

In [13]:
warehouses['city'] = warehouses['city'].fillna('Ahmedabad')

In [14]:
warehouses['city'].isna().value_counts()

city
False    55
Name: count, dtype: int64

In [ ]:
warehouses.info()

In [16]:
warehouses['warehouse_name']= warehouses['warehouse_name'].str.strip().str.title()

In [17]:
warehouses['region'] = warehouses['region'].str.strip().str.title()

In [18]:
warehouses['region'].value_counts().sum()

np.int64(55)

In [19]:
warehouses['capacity'] = warehouses['capacity'].fillna(warehouses.groupby(['region', 'state','city'])['capacity'].transform('median'))

In [20]:
warehouses['capacity'] = warehouses['capacity'].fillna(warehouses['capacity'].median())

In [21]:
warehouses['capacity'].isna().value_counts()

capacity
False    55
Name: count, dtype: int64

In [22]:
warehouses['capacity'].max()

np.float64(49866.0)

In [23]:
# outlier remove
min_capacity = warehouses['capacity'].quantile(0.25)
max_capacity = warehouses['capacity'].quantile(0.75)

iqr_capacity = max_capacity - min_capacity

lower_capacity = min_capacity - 1.5 * iqr_capacity
higher_capacity = max_capacity + 1.5 * iqr_capacity

outlier_capacity = warehouses[
    (warehouses['capacity'] < lower_capacity) | 
    (warehouses['capacity'] > higher_capacity) 
]

warehouses = warehouses[
    (warehouses['capacity'] >= lower_capacity) & 
    (warehouses['capacity'] <= higher_capacity) 
]


In [24]:
warehouses = warehouses[warehouses['capacity'] >  0]

In [25]:
warehouses['capacity'].max()

np.float64(49866.0)

In [26]:
# sqlite 
conn = sqlite3.connect('managing.db')

In [27]:
warehouses.to_sql('warehouses',conn, if_exists='replace' , index=False)

50

In [28]:
pd.read_sql_query('select * from warehouses limit 1 ', conn)

,warehouse_id,warehouse_name,city,state,region,capacity
0,WH032,Faridabad Distribution Center,Faridabad,Haryana,North,18934.0


In [29]:
# SQL Questions

# Examples:

# Beginner
# Count warehouses by state.
# Average warehouse capacity.
# Largest warehouse.
# Intermediate
# Warehouses with missing capacity.
# Warehouses having invalid capacity.
# Duplicate warehouse names.
# Warehouses by region.
# Advanced
# Capacity contribution by region.
# Rank warehouses by capacity.
# Warehouses above regional average.
# Detect wrong city-state mappings.

In [30]:
# Count warehouses by state.
pd.read_sql_query('select state, count(warehouse_id) as total_count from warehouses group by state order by total_count desc', conn)

,state,total_count
0,West Bengal,9
1,Tamil Nadu,7
2,Maharashtra,7
3,Uttar Pradesh,6
4,Rajasthan,6
5,Karnataka,4
6,Punjab,3
7,Haryana,3
8,Gujarat,3
9,Delhi,2


In [31]:
# Average warehouse capacity.
pd.read_sql_query('select warehouse_name, round(avg(capacity),2) as avg_capacity from warehouses group by warehouse_name order by avg_capacity desc ',conn)

,warehouse_name,avg_capacity
0,Ahmedabad Distribution Center,49866.00
1,Surat Distribution Center,44754.50
2,Faridabad Distribution Center,36071.00
3,Mysore Distribution Center,35692.00
4,Nagpur Distribution Center,34535.00
5,Lucknow Distribution Center,29629.50
6,Bangalore Distribution Center,28748.50
7,New Delhi Distribution Center,27524.00
8,Kolkata Distribution Center,26434.44
9,Chennai Distribution Center,25713.60


In [32]:
# Largest warehouse.
pd.read_sql_query('select warehouse_name, capacity from warehouses where capacity = (select max(capacity) from warehouses)', conn)

,warehouse_name,capacity
0,Ahmedabad Distribution Center,49866.0


In [33]:
# Warehouses with missing capacity.
pd.read_sql_query('select * from warehouses where capacity is null', conn)

,warehouse_id,warehouse_name,city,state,region,capacity


In [34]:
# Warehouses having invalid capacity.
pd.read_sql_query('select * from warehouses where capacity is null or capacity < 0 ', conn)

,warehouse_id,warehouse_name,city,state,region,capacity


In [35]:
# Duplicate warehouse names.
pd.read_sql_query('select warehouse_name , count(warehouse_name) as dupp_count from warehouses group by warehouse_name having count(warehouse_name) > 1 order by dupp_count desc', conn)

,warehouse_name,dupp_count
0,Kolkata Distribution Center,9
1,Jaipur Distribution Center,6
2,Chennai Distribution Center,5
3,Nagpur Distribution Center,4
4,Lucknow Distribution Center,4
5,Faridabad Distribution Center,4
6,New Delhi Distribution Center,3
7,Surat Distribution Center,2
8,Pune Distribution Center,2
9,Mysore Distribution Center,2


In [36]:
# Warehouses by region.
pd.read_sql_query('select region , count(warehouse_name) as warehouses_count from warehouses group by region order by warehouses_count desc', conn)

,region,warehouses_count
0,West,16
1,North,14
2,South,11
3,East,9


In [37]:
# Capacity contribution by region.
pd.read_sql_query("""
with contribution_data as (
    select sum(capacity) as total_capacity from warehouses
    )
    select region, sum(capacity) as capacity, 
    round(sum(capacity) * 100.0 / (select total_capacity from contribution_data ),2) as contribution_pct
    from warehouses
    group by region
    order by contribution_pct desc
""", conn)

,region,capacity,contribution_pct
0,West,468762.0,33.32
1,North,412901.5,29.35
2,South,287124.0,20.41
3,East,237910.0,16.91


In [ ]:
# Rank warehouses by capacity.
pd.read_sql_query('''
select warehouse_id,
    warehouse_name,
    capacity,
    dense_rank() over(order by capacity desc) as capacity_rank
    from warehouses''', conn)

In [ ]:
# Warehouses above regional average.
pd.read_sql_query(''' with regional_data as (
    select region, avg(capacity) as avg_regional_capacity
    from warehouses
    group by region
)
select 
w.warehouse_id,
w.warehouse_name,
w.region,
w.capacity
from warehouses w join regional_data r
on w.region = r.region
where w.capacity > avg_regional_capacity
''', conn)

In [40]:
warehouses.to_sql("warehouses",mysql_conn,if_exists='replace',index=False)

50

In [41]:
warehouses.to_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/warehouses.csv", index=False)